# Paradigm 1 — Explainable Recommenders Grounded in Collaborative Evidence

**Anchor paper**: XRec (Ma et al., EMNLP 2024) — LLM-based explainable recommendation

**Architecture**: Collaborative filtering (BPR-MF) for ranking + LLM for explanation generation.
The collaborative model captures user-item interaction patterns; the LLM generates
natural-language explanations grounded in the user's history and item metadata.

**Key insight from XRec**: User/item embeddings from a collaborative encoder (LightGCN)
are projected into the LLM's input space via lightweight adapters, enabling the LLM
to generate explanations conditioned on collaborative signals.

**Our implementation**: We use BPR-MF for ranking (simpler, no GNN overhead) and
prompt an LLM with the user's reading history + item metadata for explanations.
Both API-based (OpenAI/Claude) and local LLaMA options are provided.

**Ranking metrics**: HR@K, NDCG@K, Recall@K  
**Explanation metrics**: Relevance, Specificity, Hallucination rate

In [ ]:
!pip install -q torch numpy pandas implicit

In [ ]:
import sys
sys.path.insert(0, '.')

import pickle
import numpy as np
from collections import defaultdict
from tutorial_utils import (
    evaluate_ranking, evaluate_explanations,
    save_results, save_predictions, LatencyTracker,
    DATA_DIR, RESULTS_DIR
)

# Load shared data
with open(DATA_DIR / 'shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)

user_histories = shared['user_histories']
test_ground_truth = shared['test_ground_truth']
val_ground_truth = shared['val_ground_truth']
item_titles = shared['item_titles']
all_items = shared['all_items']
n_users = shared['n_users']
n_items = shared['n_items']

print(f"Users: {n_users}, Items: {n_items}")
print(f"Test users: {len(test_ground_truth)}")

## 1. Collaborative Filtering: BPR Matrix Factorization

BPR-MF learns user and item embeddings by optimizing pairwise ranking loss.
This gives us the recommendation component — the LLM handles explanations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


class BPR_MF(nn.Module):
    """Bayesian Personalized Ranking with Matrix Factorization."""

    def __init__(self, n_users, n_items, embedding_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, embedding_dim)
        self.item_emb = nn.Embedding(n_items, embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, user_idx, pos_idx, neg_idx):
        u = self.user_emb(user_idx)
        p = self.item_emb(pos_idx)
        n = self.item_emb(neg_idx)
        pos_score = (u * p).sum(dim=1)
        neg_score = (u * n).sum(dim=1)
        return pos_score, neg_score

    def predict(self, user_idx):
        """Score all items for a user."""
        u = self.user_emb(user_idx)
        scores = u @ self.item_emb.weight.T
        return scores


def bpr_loss(pos_score, neg_score):
    return -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

In [ ]:
# Build training triples: (user, positive_item, negative_item)
user_item_set = set()
for u, items in user_histories.items():
    for it in items:
        user_item_set.add((u, it))

all_items_list = list(range(n_items))


def sample_batch(batch_size, rng):
    users, pos_items, neg_items = [], [], []
    user_list = list(user_histories.keys())
    for _ in range(batch_size):
        u = rng.choice(user_list)
        pos = rng.choice(user_histories[u])
        neg = rng.choice(all_items_list)
        while (u, neg) in user_item_set:
            neg = rng.choice(all_items_list)
        users.append(u)
        pos_items.append(pos)
        neg_items.append(neg)
    return (
        torch.LongTensor(users),
        torch.LongTensor(pos_items),
        torch.LongTensor(neg_items),
    )

In [ ]:
# Train BPR-MF
EMBEDDING_DIM = 64
LR = 1e-3
N_EPOCHS = 30
BATCH_SIZE = 2048
STEPS_PER_EPOCH = 500

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BPR_MF(n_users, n_items, EMBEDDING_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
rng = np.random.RandomState(42)

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0
    for step in range(STEPS_PER_EPOCH):
        users, pos, neg = sample_batch(BATCH_SIZE, rng)
        users, pos, neg = users.to(device), pos.to(device), neg.to(device)
        pos_score, neg_score = model(users, pos, neg)
        loss = bpr_loss(pos_score, neg_score)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{N_EPOCHS}, Loss: {total_loss / STEPS_PER_EPOCH:.4f}")

## 2. Generate recommendations (ranking)

In [ ]:
K = 20
tracker = LatencyTracker()

model.eval()
predictions = {}

with torch.no_grad():
    for user_idx in test_ground_truth:
        tracker.start()
        scores = model.predict(torch.LongTensor([user_idx]).to(device)).squeeze()
        # Mask out items already in history
        history = user_histories.get(user_idx, [])
        for it in history:
            scores[it] = -float('inf')
        top_k = torch.topk(scores, K).indices.cpu().tolist()
        predictions[user_idx] = top_k
        tracker.stop()

# Evaluate ranking
ranking_results = evaluate_ranking(predictions, test_ground_truth, k_values=[5, 10, 20])
latency = tracker.summary()

print("Ranking results:")
for metric, val in ranking_results.items():
    print(f"  {metric}: {val:.4f}")
print(f"\nLatency: {latency['mean_latency_ms']:.2f} ms/user")

## 3. Generate explanations with LLM

For each recommended item, we generate an explanation grounded in:
- The user's reading history (collaborative evidence)
- The recommended item's title (item metadata)

### Option A: API-based (OpenAI)

In [ ]:
EXPLANATION_PROMPT = """You are a book recommendation system. A user has read the following books:
{history}

You recommended: "{recommended_title}"

Explain in 2-3 sentences why this book is a good recommendation for this user.
Be specific — reference concrete titles from their history and explain the connection.
Do not mention any books that are not in the user's history or the recommendation."""


def format_explanation_prompt(user_idx, item_idx):
    history = user_histories.get(user_idx, [])
    history_titles = [item_titles.get(it, '?') for it in history[-10:]]  # last 10
    rec_title = item_titles.get(item_idx, '?')
    return EXPLANATION_PROMPT.format(
        history='\n'.join(f'- {t}' for t in history_titles),
        recommended_title=rec_title,
    )

In [ ]:
# --- Option A: OpenAI API ---
# Uncomment and set your API key to use this option.

# import openai
# client = openai.OpenAI(api_key="YOUR_KEY")
#
# def generate_explanation_api(user_idx, item_idx):
#     prompt = format_explanation_prompt(user_idx, item_idx)
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=150,
#         temperature=0.3,
#     )
#     return response.choices[0].message.content.strip()

### Option B: Local LLaMA (Colab / GPU)

In [ ]:
# --- Option B: Local LLaMA with 4-bit quantization ---
# Requires GPU. On Colab Pro (A100) or local GPU with >= 8GB VRAM.

# !pip install -q transformers accelerate bitsandbytes
#
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )
#
# MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"  # requires HF access token
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, quantization_config=bnb_config, device_map="auto"
# )
#
# def generate_explanation_local(user_idx, item_idx):
#     prompt = format_explanation_prompt(user_idx, item_idx)
#     inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
#     with torch.no_grad():
#         outputs = llm.generate(
#             **inputs, max_new_tokens=150, temperature=0.3, do_sample=True
#         )
#     response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
#     return response.strip()

In [ ]:
# Generate explanations for a sample of test users
# Switch between generate_explanation_api / generate_explanation_local

N_EXPLAIN = 100  # number of users to generate explanations for
explain_tracker = LatencyTracker()
explanations = {}

sample_users = list(test_ground_truth.keys())[:N_EXPLAIN]

# Placeholder: replace with your chosen generate function
def generate_explanation(user_idx, item_idx):
    """Replace this with generate_explanation_api or generate_explanation_local."""
    # For now, return the prompt itself as a placeholder
    history = user_histories.get(user_idx, [])
    history_titles = [item_titles.get(it, '?') for it in history[-5:]]
    rec_title = item_titles.get(item_idx, '?')
    return (f"Since you enjoyed {history_titles[-1] if history_titles else 'similar books'}, "
            f"you might like '{rec_title}' which shares similar themes and style.")


for user_idx in sample_users:
    top_item = predictions[user_idx][0]  # top-1 recommendation
    explain_tracker.start()
    text = generate_explanation(user_idx, top_item)
    explain_tracker.stop()
    explanations[user_idx] = {"item_idx": top_item, "text": text}

print(f"Generated {len(explanations)} explanations")
print(f"\nSample explanation for user {sample_users[0]}:")
print(f"  Recommended: {item_titles.get(predictions[sample_users[0]][0], '?')}")
print(f"  Explanation: {explanations[sample_users[0]]['text']}")

## 4. Evaluate explanations

In [ ]:
all_titles_set = set(item_titles.values())
explanation_results = evaluate_explanations(
    explanations, user_histories, item_titles, all_titles_set
)

print("Explanation quality:")
for metric, val in explanation_results.items():
    print(f"  {metric}: {val:.4f}")

explain_latency = explain_tracker.summary()
print(f"\nExplanation latency: {explain_latency['mean_latency_ms']:.2f} ms/explanation")

## 5. Save results

In [ ]:
results = {
    "paradigm": "explainable",
    "model": "BPR-MF + LLM explanation",
    "anchor_paper": "XRec (Ma et al., EMNLP 2024)",
    "ranking": ranking_results,
    "explanation": explanation_results,
    "system": {
        "ranking_latency": latency,
        "explanation_latency": explain_latency,
    },
}

save_results("explainable", results)
save_predictions("explainable", predictions)
print("Done.")